# Setup


In [ ]:
%pip install -q -U "crewai[google-genai,tools]"

# Auth & Model


In [ ]:
import os
from google.colab import userdata

GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")

if not GEMINI_API_KEY:
    raise ValueError(
        "Please add GEMINI_API_KEY to Google Colab Secrets."
    )

os.environ["GEMINI_API_KEY"] = GEMINI_API_KEY

from crewai import Agent, Task, Crew, Process, LLM

llm = LLM(
    model="gemini/gemini-3.5-flash-lite",
    api_key=GEMINI_API_KEY,
    temperature=0.2
)

print("Gemini configured successfully!")

# Transcript


In [ ]:
transcript = """
Sprint Review & Database Migration Sync

Elena: We need to complete the primary database cutover to PostgreSQL 16 by November 14.

Marcus: I will finish the migration scripts and run load tests by November 8.

Siddharth: I will prepare the read-replicas and verify connection pool configurations by November 10.

Elena: We agreed to use zero-downtime replication with pg_logical for the cutover.

Marcus: We also need health check monitors and rollback alerts configured in Datadog.

Siddharth: I will share the replica latency metrics with Marcus by November 10.

Elena: Our final review meeting will be on November 12 to approve the maintenance window.

Meeting adjourned.
"""

with open("transcript.txt", "w", encoding="utf-8") as f:
    f.write(transcript)

print("Meeting transcript created successfully!")

# File tools


In [ ]:
from crewai_tools import FileReadTool, FileWriterTool

file_reader = FileReadTool(file_path="transcript.txt")
file_writer = FileWriterTool()

print("File tools initialized!")

# Agents


In [ ]:
from crewai import Agent

summarizer = Agent(
    role="Discussion Summarizer",
    goal="Summarize the meeting transcript in exactly five clear bullet points.",
    backstory="You specialize in distilling team discussions down to their essential takeaways without omitting context.",
    llm=llm,
    verbose=True
)

action_finder = Agent(
    role="Action Item Tracker",
    goal="Extract every action item, assignee, and delivery deadline stated in the transcript.",
    backstory="You rigorously track project deliverables and never invent assignees or missing due dates.",
    llm=llm,
    verbose=True
)

decision_logger = Agent(
    role="Decision Recorder",
    goal="Identify all decisions formally approved during the meeting.",
    backstory="You capture concrete team decisions and clearly differentiate settled conclusions from ongoing debate.",
    llm=llm,
    verbose=True
)

print("Three specialist agents created successfully!")

# Tasks


In [ ]:
from crewai import Task

summary_task = Task(
    description="""
    Read the following meeting transcript:

    {transcript}

    Summarize the main discussion in exactly five concise bullet points.
    Include important dates, project goals, and key updates.
    Do not add information that is not in the transcript.
    """,
    expected_output="Exactly five concise bullet points summarizing the meeting.",
    agent=summarizer,
    async_execution=True
)

action_task = Task(
    description="""
    Analyze the meeting transcript:

    {transcript}

    Extract every action item. For each item, identify:
    - Task
    - Responsible person
    - Deadline

    If an owner or deadline is not stated, write 'Not specified'.
    Do not invent information.
    """,
    expected_output="A structured list of action items, owners, and deadlines.",
    agent=action_finder,
    async_execution=True
)

decision_task = Task(
    description="""
    Analyze the meeting transcript:

    {transcript}

    List the decisions explicitly agreed upon.
    Distinguish confirmed decisions from suggestions.
    If there are no confirmed decisions, state that clearly.
    """,
    expected_output="A concise list of confirmed meeting decisions.",
    agent=decision_logger,
    async_execution=True
)

print("Three parallel tasks created successfully!")

# Formatter agent


In [ ]:
formatter = Agent(
    role="Notes Formatter",
    goal="Combine the meeting summary, action items, and decisions into a clear Markdown report.",
    backstory="You create accurate, professional meeting minutes that are easy to read and act upon.",
    llm=llm,
    verbose=True
)

print("Formatter agent created successfully!")

# Format task


In [ ]:
format_task = Task(
    description="""
    Create complete meeting notes using the outputs from the
    previous tasks provided in your task context.

    Combine the meeting summary, action items, and decisions into
    one clear, professional Markdown report.

    Organize the report using these headings:

    # Meeting Notes
    ## Executive Summary
    ## Key Discussion Points
    ## Action Items
    ## Decisions Made
    ## Upcoming Milestones

    Requirements:
    - Use a Markdown table for action items.
    - Include task, owner, and deadline in the table.
    - Preserve all facts and dates from the transcript.
    - Do not invent missing information.
    - Avoid repeating the same information unnecessarily.
    """,
    expected_output="A complete, well-formatted Markdown meeting notes report.",
    agent=formatter,
    context=[summary_task, action_task, decision_task],
    output_file="notes.md"
)

# Crew


In [ ]:
from crewai import Crew, Process

meeting_crew = Crew(
    agents=[
        summarizer,
        action_finder,
        decision_logger,
        formatter
    ],
    tasks=[
        summary_task,
        action_task,
        decision_task,
        format_task
    ],
    process=Process.sequential,
    verbose=True
)

print("Meeting Notes Crew created successfully!")

# Run


In [ ]:
result = await meeting_crew.kickoff_async(
    inputs={"transcript": transcript}
)

print("\nMeeting Notes Crew completed!")
print(result)

# Review


In [ ]:
from pathlib import Path
from IPython.display import display, Markdown

report_path = Path("notes.md")

if report_path.exists() and report_path.stat().st_size > 0:
    report = report_path.read_text(encoding="utf-8")

    print("Report created successfully!")
    print("Location:", report_path.resolve())
    print("Size:", report_path.stat().st_size, "bytes")

    display(Markdown(report))
else:
    print("notes.md is missing or empty.")
    print("Check the execution logs in Cell 10.")

# Download


In [ ]:
from google.colab import files

files.download("notes.md")